In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.feature import hog
import joblib
import os

MODEL_PATH     = '../models/svm_model.pkl'
IMG_SIZE       = (64, 64)
PROB_THRESHOLD = 0.70   # Minimalny próg pewności do zaakceptowania detekcji

svm_model = joblib.load(MODEL_PATH)


def extract_hog_features(img_roi):
    resized  = cv2.resize(img_roi, IMG_SIZE)
    gray     = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)
    features = hog(gray, orientations=9, pixels_per_cell=(8, 8),
                   cells_per_block=(2, 2), block_norm='L2-Hys', visualize=False)
    return features.reshape(1, -1)


def detect_and_classify_signs(image_path):
    img = cv2.imread(image_path)
    if img is None:
        print(f"[BŁĄD] Nie można wczytać: {image_path}")
        return

    img_copy = img.copy()

    # --- Etap 1: Preprocessing ---
    # Blur spójny z treningiem – trening również używał GaussianBlur(5,5)
    img_blurred = cv2.GaussianBlur(img, (5, 5), 0)
    hsv = cv2.cvtColor(img_blurred, cv2.COLOR_BGR2HSV)

    # --- Etap 2: Maska kolorów ---
    # Progi S/V celowo wyższe niż w treningu (70/50 vs 30/40) – redukcja false positives
    # na zdjęciach drogowych, gdzie tło jest bardziej zróżnicowane niż w GTSRB
    lower_red1   = np.array([0,   70,  50])
    upper_red1   = np.array([10,  255, 255])
    lower_red2   = np.array([160, 70,  50])
    upper_red2   = np.array([180, 255, 255])
    lower_yellow = np.array([20,  100, 100])
    upper_yellow = np.array([35,  255, 255])

    mask_red1     = cv2.inRange(hsv, lower_red1,   upper_red1)
    mask_red2     = cv2.inRange(hsv, lower_red2,   upper_red2)
    mask_yellow   = cv2.inRange(hsv, lower_yellow, upper_yellow)
    mask_combined = mask_red1 + mask_red2 + mask_yellow

    # --- Etap 3: Morfologia ---
    # OPEN 3x3 usuwa szum, CLOSE 7x7 wypełnia dziury w masce
    kernel_open  = np.ones((3, 3), np.uint8)
    kernel_close = np.ones((7, 7), np.uint8)
    mask_clean = cv2.morphologyEx(mask_combined, cv2.MORPH_OPEN,  kernel_open)
    mask_clean = cv2.morphologyEx(mask_clean,    cv2.MORPH_CLOSE, kernel_close)

    contours, _ = cv2.findContours(mask_clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # --- Etap 4: Filtracja geometryczna ---
    potential_candidates = []

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < 600:
            continue

        x, y, w, h = cv2.boundingRect(cnt)
        aspect_ratio = w / float(h)
        if not (0.7 < aspect_ratio < 1.3):
            continue

        hull      = cv2.convexHull(cnt)
        hull_area = cv2.contourArea(hull)
        if hull_area == 0:
            continue
        solidity = float(area) / hull_area

        # Żółte (trójkąty) i czerwone mają różną charakterystykę kształtu
        roi_yellow = mask_yellow[y:y+h, x:x+w]
        roi_red    = mask_red1[y:y+h, x:x+w] + mask_red2[y:y+h, x:x+w]

        if cv2.countNonZero(roi_yellow) > cv2.countNonZero(roi_red):
            if solidity < 0.80:
                continue
        else:
            if solidity < 0.20:
                continue

        potential_candidates.append(cnt)

    # --- Etap 5: Klasyfikacja ---
    detected = 0
    for cnt in potential_candidates:
        x, y, w, h = cv2.boundingRect(cnt)
        margin = int(0.05 * w)
        x1 = max(0, x - margin)
        y1 = max(0, y - margin)
        x2 = min(img.shape[1], x + w + margin)
        y2 = min(img.shape[0], y + h + margin)

        roi = img[y1:y2, x1:x2]
        if roi.shape[0] <= 10 or roi.shape[1] <= 10:
            continue

        features      = extract_hog_features(roi)
        probabilities = svm_model.predict_proba(features)[0]
        max_prob      = np.max(probabilities)
        prediction    = svm_model.predict(features)[0]

        if max_prob < PROB_THRESHOLD:
            print(f"[ODRZUCONO] {prediction} pewność={max_prob*100:.1f}% (próg {PROB_THRESHOLD*100:.0f}%)")
            continue

        detected += 1
        cv2.rectangle(img_copy, (x1, y1), (x2, y2), (0, 255, 0), 3)
        label = f"{prediction} ({max_prob*100:.1f}%)"
        cv2.putText(img_copy, label, (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    name = os.path.basename(image_path)
    if detected == 0:
        print(f"[INFO] {name}: nie wykryto zadnego znaku (kandydatow geom.: {len(potential_candidates)})")

    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(img_copy, cv2.COLOR_BGR2RGB))
    plt.title(f"{name} — wykryto: {detected}")
    plt.axis('off')
    plt.show()


# Testowanie - wczytuje wszystkie obrazy z folderu (dowolne nazwy, .jpg/.jpeg/.png)
TEST_DIR = '../data/test_images'
test_files = sorted(f for f in os.listdir(TEST_DIR)
                    if f.lower().endswith(('.jpg', '.jpeg', '.png')))
print(f"Znaleziono {len(test_files)} zdjec testowych.")
for fn in test_files:
    detect_and_classify_signs(os.path.join(TEST_DIR, fn))

In [ ]:
# Wizualizacja krokow detekcji na jednym zdjeciu (podglad etapow + wynik koncowy)
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.feature import hog
import joblib

svm_model = joblib.load('../models/svm_model.pkl')
IMG_SIZE = (64, 64)


def show_step(title, image, is_gray=False):
    plt.figure(figsize=(8, 6))
    plt.imshow(image if is_gray else cv2.cvtColor(image, cv2.COLOR_BGR2RGB),
               cmap='gray' if is_gray else None)
    plt.title(title)
    plt.axis('off')
    plt.show()


def extract_hog_features(img_roi):
    gray = cv2.cvtColor(cv2.resize(img_roi, IMG_SIZE), cv2.COLOR_BGR2GRAY)
    return hog(gray, orientations=9, pixels_per_cell=(8, 8),
               cells_per_block=(2, 2), block_norm='L2-Hys').reshape(1, -1)


def detect_and_classify_signs_v3(image_path, show_steps=True):
    img = cv2.imread(image_path)
    if img is None:
        print(f"Blad wczytania obrazu: {image_path}")
        return
    img_copy = img.copy()

    hsv = cv2.cvtColor(cv2.GaussianBlur(img, (5, 5), 0), cv2.COLOR_BGR2HSV)
    mask_red1 = cv2.inRange(hsv, np.array([0, 70, 50]),   np.array([10, 255, 255]))
    mask_red2 = cv2.inRange(hsv, np.array([160, 70, 50]), np.array([180, 255, 255]))
    mask_yellow = cv2.inRange(hsv, np.array([20, 100, 100]), np.array([35, 255, 255]))
    mask_combined = mask_red1 | mask_red2 | mask_yellow

    mask_open = cv2.morphologyEx(mask_combined, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    mask_clean = cv2.morphologyEx(mask_open, cv2.MORPH_CLOSE, np.ones((7, 7), np.uint8))

    if show_steps:
        show_step("1. Oryginal", img)
        show_step("2. Maska czerwona", mask_red1 | mask_red2, is_gray=True)
        show_step("3. Maska zolta", mask_yellow, is_gray=True)
        show_step("4. Maska polaczona", mask_combined, is_gray=True)
        show_step("5. Morfologia: otwarcie", mask_open, is_gray=True)
        show_step("6. Morfologia: zamkniecie", mask_clean, is_gray=True)

    contours, _ = cv2.findContours(mask_clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    candidates = []
    for cnt in contours:
        if cv2.contourArea(cnt) < 600:
            continue
        x, y, w, h = cv2.boundingRect(cnt)
        if not (0.7 < w / float(h) < 1.3):
            continue
        hull_area = cv2.contourArea(cv2.convexHull(cnt))
        if hull_area == 0:
            continue
        solidity = cv2.contourArea(cnt) / hull_area
        roi_yellow = mask_yellow[y:y+h, x:x+w]
        roi_red = mask_red1[y:y+h, x:x+w] | mask_red2[y:y+h, x:x+w]
        if cv2.countNonZero(roi_yellow) > cv2.countNonZero(roi_red):
            if solidity < 0.80:
                continue
        else:
            if solidity < 0.20:
                continue
        candidates.append(cnt)

    if show_steps:
        img_candidates = img.copy()
        for cnt in candidates:
            x, y, w, h = cv2.boundingRect(cnt)
            cv2.rectangle(img_candidates, (x, y), (x + w, y + h), (255, 0, 0), 2)
        show_step(f"7. Kandydaci ({len(candidates)})", img_candidates)

    for cnt in candidates:
        x, y, w, h = cv2.boundingRect(cnt)
        m = int(0.05 * w)
        x1, y1 = max(0, x - m), max(0, y - m)
        x2, y2 = min(img.shape[1], x + w + m), min(img.shape[0], y + h + m)
        roi = img[y1:y2, x1:x2]
        if roi.shape[0] <= 10 or roi.shape[1] <= 10:
            continue
        feat = extract_hog_features(roi)
        proba = svm_model.predict_proba(feat)[0]
        if proba.max() > 0.70:
            label = f"{svm_model.predict(feat)[0]} ({proba.max() * 100:.1f}%)"
            cv2.rectangle(img_copy, (x1, y1), (x2, y2), (0, 255, 0), 3)
            cv2.putText(img_copy, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    show_step("8. Wynik koncowy", img_copy)


detect_and_classify_signs_v3('../data/test_images/17.jpg', show_steps=True)

In [ ]:
# ============================================================
# EWALUACJA SKUTECZNOSCI DETEKCJI na zdjeciach testowych
# >>> UZUPELNIJ slownik GROUND_TRUTH: nazwa pliku -> faktyczny znak <<<
# Dozwolone klasy: Droga_z_pierwszenstwem, Ograniczenie_30, Stop,
#                  Ustap_pierwszenstwa, Zakaz_wjazdu.  Puste '' = pomijane.
# ============================================================
import os, cv2, numpy as np, joblib
import matplotlib.pyplot as plt
from skimage.feature import hog
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

TEST_DIR       = '../data/test_images'
PROB_THRESHOLD = 0.70
CLASSES = ['Droga_z_pierwszenstwem', 'Ograniczenie_30', 'Stop',
           'Ustap_pierwszenstwa', 'Zakaz_wjazdu']

GROUND_TRUTH = {
    '1.jpg': '',  '2.jpg': '',  '3.jpg': '',  '4.jpg': '',  '5.jpg': '',
    '6.jpg': '',  '7.jpg': '',  '8.jpg': '',  '9.jpg': '',  '10.jpg': '',
    '11.jpg': '', '12.jpg': '', '13.jpg': 'Ustap_pierwszenstwa', '14.jpg': '',
    '15.jpg': 'Ograniczenie_30', '16.jpg': 'Ograniczenie_30',
    '17.jpg': 'Ograniczenie_30', '18.jpg': '', '19.jpg': '', '20.jpg': '',
    '21.jpg': '', '22.jpg': 'Droga_z_pierwszenstwem', '23.jpg': '',
    '24.jpg': '', '25.jpg': 'Droga_z_pierwszenstwem',
}

svm_model = joblib.load('../models/svm_model.pkl')


def detect_top(image_path):
    # zwraca (klasa, pewnosc) najpewniejszej detekcji powyzej progu, albo None
    img = cv2.imread(image_path)
    if img is None:
        return None
    hsv = cv2.cvtColor(cv2.GaussianBlur(img, (5, 5), 0), cv2.COLOR_BGR2HSV)
    m1 = cv2.inRange(hsv, np.array([0, 70, 50], np.uint8),   np.array([10, 255, 255], np.uint8))
    m2 = cv2.inRange(hsv, np.array([160, 70, 50], np.uint8), np.array([180, 255, 255], np.uint8))
    my = cv2.inRange(hsv, np.array([20, 100, 100], np.uint8), np.array([35, 255, 255], np.uint8))
    mask = m1 | m2 | my
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  np.ones((3, 3), np.uint8))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((7, 7), np.uint8))
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    best = None
    for cnt in cnts:
        area = cv2.contourArea(cnt)
        if area < 600:
            continue
        x, y, w, h = cv2.boundingRect(cnt)
        if not (0.7 < w / float(h) < 1.3):
            continue
        ha = cv2.contourArea(cv2.convexHull(cnt))
        if ha == 0:
            continue
        sol = area / ha
        yel = cv2.countNonZero(my[y:y+h, x:x+w])
        red = cv2.countNonZero(cv2.bitwise_or(m1, m2)[y:y+h, x:x+w])
        if (yel > red and sol < 0.80) or (yel <= red and sol < 0.20):
            continue
        mg = int(0.05 * w)
        x1, y1 = max(0, x - mg), max(0, y - mg)
        x2, y2 = min(img.shape[1], x + w + mg), min(img.shape[0], y + h + mg)
        roi = img[y1:y2, x1:x2]
        if roi.shape[0] <= 10 or roi.shape[1] <= 10:
            continue
        gray = cv2.cvtColor(cv2.resize(roi, (64, 64)), cv2.COLOR_BGR2GRAY)
        feat = hog(gray, orientations=9, pixels_per_cell=(8, 8),
                   cells_per_block=(2, 2), block_norm='L2-Hys').reshape(1, -1)
        p = svm_model.predict_proba(feat)[0]
        if p.max() >= PROB_THRESHOLD and (best is None or p.max() > best[1]):
            best = (svm_model.predict(feat)[0], p.max())
    return best


y_true, y_pred = [], []
for fn, true_cls in GROUND_TRUTH.items():
    if not true_cls:
        continue
    res = detect_top(os.path.join(TEST_DIR, fn))
    y_true.append(true_cls)
    y_pred.append(res[0] if res else 'BRAK')

if not y_true:
    print('Uzupelnij slownik GROUND_TRUTH (nazwa pliku -> klasa), potem odpal ponownie.')
else:
    n        = len(y_true)
    detected = sum(1 for p in y_pred if p != 'BRAK')
    correct  = sum(1 for t, p in zip(y_true, y_pred) if t == p)
    print('Zdjec ocenionych:           %d' % n)
    print('Wykryto cokolwiek:          %d/%d (%.0f%%)' % (detected, n, detected / n * 100))
    print('Poprawna detekcja+klasa:    %d/%d (%.0f%%)' % (correct, n, correct / n * 100))

    labels = CLASSES + ['BRAK']
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    fig, ax = plt.subplots(figsize=(9, 8))
    disp.plot(cmap='Blues', ax=ax, xticks_rotation=45, values_format='d')
    plt.title('Skutecznosc detekcji na zdjeciach testowych\n(kolumna BRAK = nie wykryto znaku)',
              fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()